In [408]:
#%pip install itables

## Bronze Transformation

The Bronze transformation ingests the raw Victorian EV charging-location CSV file and stores it as a managed Delta Lake table. It preserves the source data with minimal processing, providing a queryable foundation for the Silver transformation.

The transformation includes:

- Reading the CSV from the configured volume with headers and inferred data types.
- Writing the records to a managed Bronze Delta table.
- Replacing the existing table and schema when the pipeline is rerun.
- Verifying the resulting schema and record count.
- Displaying a sample of the ingested data in the notebook.

The Bronze layer intentionally performs minimal cleansing or business-rule validation. Data-type enforcement, standardisation, deduplication and quality filtering are applied during the Silver transformation.                                                                                                                                                                                                                    

In [410]:
# Install ITables once if it is not already available.
# Uncomment and run this in a separate notebook cell if required:
# %pip install itables

from itables import show

from pyspark.sql import SparkSession
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

# Retrieve the active Spark session or create one.
spark = SparkSession.builder.getOrCreate()

# Define the raw source file name and location.
volume_raw_ev_charging_locations = (
    "/Volumes/transport_planning/default/ev_charging_locations/"
    "victoria_synthetic_ev_charging_locations_2026.csv"
)

# Define the bronze table name and location.
table_bronze_ev_charging_locations = (
    "transport_planning.default."
    "bronze_victoria_synthetic_ev_charging_locations_2026"
)


# Define the expected structure of the raw source file (CSV).
bronze_ev_charging_schema = StructType([
    StructField("location_id", StringType(), True),
    StructField("site_name", StringType(), True),
    StructField("suburb_locality", StringType(), True),

    # Postcode is an identifier, so store it as text rather than a number.
    StructField("postcode", StringType(), True),

    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("region", StringType(), True),
    StructField("population_centre_type", StringType(), True),
    StructField("site_type", StringType(), True),
    StructField("charger_count", IntegerType(), True),
    StructField("charging_bays", IntegerType(), True),
    StructField("max_power_kw", IntegerType(), True),
    StructField("charger_category", StringType(), True),
    StructField("connector_types", StringType(), True),
    StructField("access_type", StringType(), True),
    StructField("availability", StringType(), True),
    StructField("payment_methods", StringType(), True),
    StructField("network_operator", StringType(), True),
    StructField("location_basis", StringType(), True),
    StructField("confidence_level", StringType(), True),
    StructField("synthetic_flag", BooleanType(), True),
    StructField("generation_note", StringType(), True),
    StructField("scenario_year", IntegerType(), True),
    StructField("scenario_label", StringType(), True),
    StructField("modelled_operational_status", StringType(), True),
    StructField("charging_scope", StringType(), True),
])


# Read the CSV using the explicitly defined schema.
bronze_sdf = (
    spark.read
    .format("csv")
    .schema(bronze_ev_charging_schema)
    .option("header", True)

    # Validate the CSV header against the supplied schema.
    .option("enforceSchema", False)

    # Stop ingestion when values cannot be parsed according to the schema.
    .option("mode", "FAILFAST")

    .load(volume_raw_ev_charging_locations)
)


# Write the raw records to the managed Bronze Delta table.
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(table_bronze_ev_charging_locations)
)


# Read the persisted table so verification checks the saved result
# rather than only the source DataFrame.
bronze_result_df = spark.table(
    table_bronze_ev_charging_locations
)


# Verify the persisted schema and record count.
bronze_result_df.printSchema()

bronze_row_count = bronze_result_df.count()
print(f"Bronze rows written: {bronze_row_count:,}")


# Render a small interactive preview in the notebook UI.
show(bronze_result_df.limit(20).toPandas())

root
 |-- location_id: string (nullable = true)
 |-- site_name: string (nullable = true)
 |-- suburb_locality: string (nullable = true)
 |-- postcode: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- region: string (nullable = true)
 |-- population_centre_type: string (nullable = true)
 |-- site_type: string (nullable = true)
 |-- charger_count: integer (nullable = true)
 |-- charging_bays: integer (nullable = true)
 |-- max_power_kw: integer (nullable = true)
 |-- charger_category: string (nullable = true)
 |-- connector_types: string (nullable = true)
 |-- access_type: string (nullable = true)
 |-- availability: string (nullable = true)
 |-- payment_methods: string (nullable = true)
 |-- network_operator: string (nullable = true)
 |-- location_basis: string (nullable = true)
 |-- confidence_level: string (nullable = true)
 |-- synthetic_flag: boolean (nullable = true)
 |-- generation_note: string (nullable = true)
 |-- sc

Bronze rows written: 1,200


<!--| quarto-html-table-processing: none -->
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

 
 
 
 

 
 
 
 

 
 
 
 

 
 
 
 
 
 
 
 
 Loading ITables v2.9.1 from the internet...
 (need help ?)
 
 
 
 
 
 🔒 ⓘ location_id 
 site_name 
 suburb_locality 
 postcode 
 latitude 
 longitude 
 region 
 population_centre_type 
 site_type 
 charger_count 
 charging_bays 
 max_power_kw 
 charger_category 
 connector_types 
 access_type 
 availability 
 payment_methods 
 network_operator 
 location_basis 
 confidence_level 
 synthetic_flag 
 generation_note 
 scenario_year 
 scenario_label 
 modelled_operational_status 
 charging_scope 
 
 VIC-2026-SYN-00001 Melbourne Shopping precinct Synthetic 2026 Site 0001 Melbourne 3000 -37.810614 144.966492 Greater Melbourne Metropolitan suburb Shopping precinct 6 6 250 DC ultra-fast CCS2 Public 24/7 Contactless card; mobile app; RFID Synthetic Charge Network B synthetic_2026_market_calibrated medium True Fictional site; aggregate public DC network calibrated to 300 sites and 1,000 plugs for the April 2026 market snapshot 2026 current_market_snapshot_2026 synthetic_modelled_current public_fast_or_ultra_fast 
 VIC-2026-SYN-00002 Melbourne Hotel or accommodation Synthetic 2026 Site 0002 Melbourne 3000 -37.814759 144.965690 Greater Melbourne Metropolitan suburb Hotel or accommodation 4 4 22 AC destination Type 2 Destination / customer access Venue hours Contactless card; mobile app Municipal EV Access synthetic_2026_market_calibrated medium True Fictional AC destination/workplace site; not included in the 300-site and 1,000-plug public DC benchmark 2026 current_market_snapshot_2026 synthetic_modelled_current ac_destination_or_workplace 
 VIC-2026-SYN-00003 Melbourne Public library Synthetic 2026 Site 0003 Melbourne 3000 -37.814507 144.962225 Greater Melbourne Metropolitan suburb Public library 2 2 22 AC destination Type 2 Public 24/7 Contactless card; mobile app Regional Mobility Network synthetic_2026_market_calibrated medium True Fictional AC destination/workplace site; not included in the 300-site and 1,000-plug public DC benchmark 2026 current_market_snapshot_2026 synthetic_modelled_current ac_destination_or_workplace 
 VIC-2026-SYN-00004 Melbourne Workplace visitor parking Synthetic 2026 Site 0004 Melbourne 3000 -37.813365 144.966721 Greater Melbourne Metropolitan suburb Workplace visitor parking 2 2 22 AC destination Type 2 Public during business hours 08:00-18:00 weekdays Contactless card; mobile app Regional Mobility Network synthetic_2026_market_calibrated medium True Fictional AC destination/workplace site; not included in the 300-site and 1,000-plug public DC benchmark 2026 current_market_snapshot_2026 synthetic_modelled_current ac_destination_or_workplace 
 VIC-2026-SYN-00005 Melbourne Community facility Synthetic 2026 Site 0005 Melbourne 3000 -37.816747 144.962130 Greater Melbourne Metropolitan suburb Community facility 6 6 22 AC destination Type 2 Public 24/7 Contactless card; mobile app Regional Mobility Network synthetic_2026_market_calibrated medium True Fictional AC destination/workplace site; not included in the 300-site and 1,000-plug public DC benchmark 2026 current_market_snapshot_2026 synthetic_modelled_current ac_destination_or_workplace 
 VIC-2026-SYN-00006 Melbourne Supermarket car park Synthetic 2026 Site 0006 Melbourne 3000 -37.813424 144.961130 Greater Melbourne Metropolitan suburb Supermarket car park 4 4 180 DC ultra-fast CCS2 Public 24/7 Contactless card; mobile app; RFID Synthetic Charge Network A synthetic_2026_market_calibrated medium True Fictional site; aggregate public DC network calibrated to 300 sites and 1,000 plugs for the April 2026 market snapshot 2026 current_market_snapshot_2026 synthetic_modelled_current public_fast_or_ultra_fast 
 VIC-2026-SYN-00007 Melbourne Public library Synthetic 2026 Site 0007 Melbourne 3000 -37.814495 144.966115 Greater Melbourne Metropolitan suburb Public library 6 6 22 AC destination Type 2 Public 24/7 C

## Silver Transformation

The Silver transformation converts the raw Bronze EV charging-location data into a clean, consistent and analysis-ready dataset. It applies standardised data types, validation rules and deduplication before writing the approved business fields to the managed `silver_ev_charging_locations` Delta table. A processing timestamp is added to provide basic transformation traceability.

The transformation includes:

- Trimming leading and trailing whitespace from text fields.
- Converting empty strings into null values.
- Converting postcodes into four-character string identifiers.
- Enforcing appropriate numeric types for coordinates, charger counts, charging bays and power ratings.
- Standardising charger categories as `AC destination`, `DC fast` or `DC ultra-fast`.
- Correcting charging-bay counts when they are missing or lower than the charger count.
- Removing records without required location identifiers or names.
- Validating that postcodes follow the Victorian `3xxx` format.
- Validating that latitude and longitude fall within approximate Victorian geographic boundaries.
- Removing records with invalid or non-positive charger specifications.
- Removing records with unsupported charger categories.
- Resolving duplicate records using `location_id`.
- Excluding synthetic and testing metadata that is not required downstream.
- Retaining only approved business columns.
- Adding a `silver_processed_at` timestamp.
- Writing the validated result to the Silver Delta table with schema replacement enabled.

In [381]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from itables import show


# Retrieve the active Spark session or create one.
spark = SparkSession.builder.getOrCreate()

# Fully qualified source and destination table names.
table_bronze_ev_charging_locations = (
    "transport_planning.default."
    "bronze_victoria_synthetic_ev_charging_locations_2026"
)

table_silver_ev_charging_locations = (
    "transport_planning.default."
    "silver_ev_charging_locations"
)

# Read the Bronze Delta table.
bronze_df = spark.table(table_bronze_ev_charging_locations)

bronze_row_count = bronze_df.count()

In [382]:
# Identify all descriptive string columns that require whitespace cleanup.
business_string_columns = [
    "location_id",
    "site_name",
    "suburb_locality",
    "region",
    "population_centre_type",
    "site_type",
    "charger_category",
    "connector_types",
    "access_type",
    "availability",
    "payment_methods",
    "network_operator",
    "charging_scope",
]

clean_df = bronze_df

# Trim leading and trailing whitespace and convert empty strings to null.
for column_name in business_string_columns:
    clean_df = clean_df.withColumn(
        column_name,
        F.when(
            F.length(F.trim(F.col(column_name))) == 0,
            F.lit(None),
        ).otherwise(F.trim(F.col(column_name))),
    )

In [384]:
# Apply the Silver-layer data types.
clean_df = (
    clean_df
    # Postcodes are identifiers rather than numeric measurements.
    .withColumn(
        "postcode",
        F.lpad(F.col("postcode").cast("string"), 4, "0"),
    )
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("charger_count", F.col("charger_count").cast("integer"))
    .withColumn("charging_bays", F.col("charging_bays").cast("integer"))
    .withColumn("max_power_kw", F.col("max_power_kw").cast("integer"))
)

In [385]:
# Standardise the supported charger-category values.
clean_df = clean_df.withColumn(
    "charger_category",
    F.when(
        F.lower(F.col("charger_category")) == "ac destination",
        F.lit("AC destination"),
    )
    .when(
        F.lower(F.col("charger_category")) == "dc fast",
        F.lit("DC fast"),
    )
    .when(
        F.lower(F.col("charger_category")) == "dc ultra-fast",
        F.lit("DC ultra-fast"),
    )
    .otherwise(F.lit(None)),
)

# Charging bays cannot logically be lower than the charger count.
# If charging_bays is missing, default it to charger_count.
clean_df = clean_df.withColumn(
    "charging_bays",
    F.greatest(
        F.coalesce(F.col("charging_bays"), F.col("charger_count")),
        F.col("charger_count"),
    ).cast("integer"),
)

In [387]:
# Retain records that satisfy the principal Silver data-quality rules.
valid_df = clean_df.filter(
    # Required identifiers and location attributes.
    F.col("location_id").isNotNull()
    & F.col("site_name").isNotNull()
    & F.col("suburb_locality").isNotNull()

    # Victorian postcodes start with 3 and contain four digits.
    & F.col("postcode").rlike(r"^3[0-9]{3}$")

    # Approximate geographic bounds covering Victoria.
    & F.col("latitude").between(-39.3, -33.8)
    & F.col("longitude").between(140.8, 150.1)

    # Charger values must be positive and internally consistent.
    & (F.col("charger_count") > 0)
    & (F.col("charging_bays") >= F.col("charger_count"))
    & (F.col("max_power_kw") > 0)

    # Only supported charging categories are accepted.
    & F.col("charger_category").isin(
        "AC destination",
        "DC fast",
        "DC ultra-fast",
    )
)


In [388]:
# Resolve duplicate location IDs deterministically.
# The first non-null site name in alphabetical order is retained.
deduplication_window = (
    Window
    .partitionBy("location_id")
    .orderBy(F.col("site_name").asc_nulls_last())
)

deduplicated_df = (
    valid_df
    .withColumn(
        "_duplicate_sequence",
        F.row_number().over(deduplication_window),
    )
    .filter(F.col("_duplicate_sequence") == 1)
    .drop("_duplicate_sequence")
)


In [389]:
# Explicitly select only fields approved for downstream processing.
# This prevents synthetic/testing metadata from leaking into Silver.
silver_df = deduplicated_df.select(
    "location_id",
    "site_name",
    "suburb_locality",
    "postcode",
    "latitude",
    "longitude",
    "region",
    "population_centre_type",
    "site_type",
    "charger_count",
    "charging_bays",
    "max_power_kw",
    "charger_category",
    "connector_types",
    "access_type",
    "availability",
    "payment_methods",
    "network_operator",
    "charging_scope",
).withColumn(
    # Record when the Silver transformation was performed.
    "silver_processed_at",
    F.current_timestamp(),
)

# Write the curated data to the managed Silver Delta table.
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_silver_ev_charging_locations)
)

In [391]:
# Read the persisted table for verification.
silver_result_df = spark.table(table_silver_ev_charging_locations)

silver_row_count = silver_result_df.count()

print(f"Bronze rows:  {bronze_row_count:,}")
print(f"Silver rows:  {silver_row_count:,}")
print(f"Rows removed: {bronze_row_count - silver_row_count:,}")

silver_result_df.printSchema()

# Render a small interactive preview in the notebook UI.
show(silver_result_df.limit(20).toPandas())

Bronze rows:  1,200
Silver rows:  1,200
Rows removed: 0
root
 |-- location_id: string (nullable = true)
 |-- site_name: string (nullable = true)
 |-- suburb_locality: string (nullable = true)
 |-- postcode: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- region: string (nullable = true)
 |-- population_centre_type: string (nullable = true)
 |-- site_type: string (nullable = true)
 |-- charger_count: integer (nullable = true)
 |-- charging_bays: integer (nullable = true)
 |-- max_power_kw: integer (nullable = true)
 |-- charger_category: string (nullable = true)
 |-- connector_types: string (nullable = true)
 |-- access_type: string (nullable = true)
 |-- availability: string (nullable = true)
 |-- payment_methods: string (nullable = true)
 |-- network_operator: string (nullable = true)
 |-- charging_scope: string (nullable = true)
 |-- silver_processed_at: timestamp (nullable = true)



<!--| quarto-html-table-processing: none -->
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 

 
 
 
 

 
 
 
 

 
 
 
 

 
 
 
 
 
 
 
 
 Loading ITables v2.9.1 from the internet...
 (need help ?)
 
 
 
 
 
 🔒 ⓘ location_id 
 site_name 
 suburb_locality 
 postcode 
 latitude 
 longitude 
 region 
 population_centre_type 
 site_type 
 charger_count 
 charging_bays 
 max_power_kw 
 charger_category 
 connector_types 
 access_type 
 availability 
 payment_methods 
 network_operator 
 charging_scope 
 silver_processed_at 
 
 VIC-2026-SYN-00001 Melbourne Shopping precinct Synthetic 2026 Site 0001 Melbourne 3000 -37.810614 144.966492 Greater Melbourne Metropolitan suburb Shopping precinct 6 6 250 DC ultra-fast CCS2 Public 24/7 Contactless card; mobile app; RFID Synthetic Charge Network B public_fast_or_ultra_fast 2026-08-07 04:36:38.092591 
 VIC-2026-SYN-00002 Melbourne Hotel or accommodation Synthetic 2026 Site 0002 Melbourne 3000 -37.814759 144.965690 Greater Melbourne Metropolitan suburb Hotel or accommodation 4 4 22 AC destination Type 2 Destination / customer access Venue hours Contactless card; mobile app Municipal EV Access ac_destination_or_workplace 2026-08-07 04:36:38.092591 
 VIC-2026-SYN-00003 Melbourne Public library Synthetic 2026 Site 0003 Melbourne 3000 -37.814507 144.962225 Greater Melbourne Metropolitan suburb Public library 2 2 22 AC destination Type 2 Public 24/7 Contactless card; mobile app Regional Mobility Network ac_destination_or_workplace 2026-08-07 04:36:38.092591 
 VIC-2026-SYN-00004 Melbourne Workplace visitor parking Synthetic 2026 Site 0004 Melbourne 3000 -37.813365 144.966721 Greater Melbourne Metropolitan suburb Workplace visitor parking 2 2 22 AC destination Type 2 Public during business hours 08:00-18:00 weekdays Contactless card; mobile app Regional Mobility Network ac_destination_or_workplace 2026-08-07 04:36:38.092591 
 VIC-2026-SYN-00005 Melbourne Community facility Synthetic 2026 Site 0005 Melbourne 3000 -37.816747 144.962130 Greater Melbourne Metropolitan suburb Community facility 6 6 22 AC destination Type 2 Public 24/7 Contactless card; mobile app Regional Mobility Network ac_destination_or_workplace 2026-08-07 04:36:38.092591 
 VIC-2026-SYN-00006 Melbourne Supermarket car park Synthetic 2026 Site 0006 Melbourne 3000 -37.813424 144.961130 Greater Melbourne Metropolitan suburb Supermarket car park 4 4 180 DC ultra-fast CCS2 Public 24/7 Contactless card; mobile app; RFID Synthetic Charge Network A public_fast_or_ultra_fast 2026-08-07 04:36:38.092591 
 VIC-2026-SYN-00007 Melbourne Public library Synthetic 2026 Site 0007 Melbourne 3000 -37.814495 144.966115 Greater Melbourne Metropolitan suburb Public library 6 6 22 AC destination Type 2 Public 24/7 Contactless card; mobile app Synthetic Charge Network A ac_destination_or_workplace 2026-08-07 04:36:38.092591 
 VIC-2026-SYN-00008 Melbourne Workplace visitor parking Synthetic 2026 Site 0008 Melbourne 3000 -37.813943 144.964936 Greater Melbourne Metropolitan suburb Workplace visitor parking 2 2 22 AC destination Type 2 Public during business hours 08:00-18:00 weekdays Contactless card; mobile app Regional Mobility Network ac_destination_or_workplace 2026-08-07 04:36:38.092591 
 VIC-2026-SYN-00009 Melbourne Public library Synthetic 2026 Site 0009 Melbourne 3000 -37.814738 144.965326 Greater Melbourne Metropolitan suburb Public library 2 2 22 AC destination Type 2 Public 06:00-22:00 daily Contactless card; mobile app Community Charging Cooperative ac_destination_or_workplace 2026-08-07 04:36:38.092591 
 VIC-2026-SYN-00010 Melbourne Hotel or accommodation Synthetic 2026 Site 0010 Melbourne 3000 -37.814772 144.966762 Greater Melbourne Metropolitan suburb Hotel or accommodation 4 4 22 AC destination Type 2 Destination / customer access Venue hours Contactless card; mobile app Community Charging Cooperative ac_destination_or_workplace 2026-08-07 04:36:38.092591 
 (10 more rows not shown)